## Function for dictionary comparison
### This file focus on the following
1. Define a function for cross-sectional dictionary comparison
2. Define a function for calculate date differences using assumptions:
- N-1
- Median N
- beta-random (based on N-1)
(see paper)


cohort = 40,963 
nhsd = 40,102

861 cohort members not in NHSD.


In [ ]:
# load in the 2 json files 
import json
import pandas as pd

with open('cohort_geo.json', 'r') as json_a:
    dict1 = json.load(json_a) # length = 40963
    
    
with open('nhsd_geo.json', 'r') as json_b:
    dict2 = json.load(json_b) # length = 40102



In [ ]:
ids_in_cohort  =set(dict1.keys())
ids_in_nhsd = set(dict2.keys())

only_cohort = ids_in_cohort - ids_in_nhsd
rows = []
for unique_id in only_cohort:
    rows.append({
        'llc_0002_stud_id': unique_id,
    })
df_flag = pd.DataFrame(rows)
    

In [ ]:
# Flagging individuals not present in NHSD records (Has geoconsent, linked to LLC, but did not have NHSD service use record)
# to link/describe
df_flag

In [ ]:
df_flag.to_csv("no_nhsd_record.csv")

In [ ]:
# Compare dictionaries and store results in a Dataframe
# Unique ID is set as the keys for each dictionary
# dict a = cohorts
# dict b = NHSD
# gets all matching address, date comparisons

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime


In [ ]:
# version 2 - consider multiple, not just first

# Compare dictionaries and store results in a Dataframe
# Unique ID is set as the keys for each dictionary
# dict a = cohorts
# dict b = NHSD
# gets all matching address, date comparisons

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

def compare_dicts_to_df(dict_a, dict_b):
    data = []
    
    # Get unique IDs
    shared_ids = set(dict_a.keys()) & set(dict_b.keys())
    
    # Initialise variables 
    for unique_id in shared_ids:
            min_date_diff = None
            best_match_row = None
  
            # Check [ANY] matching addresses within shared IDs
            for address_a, date_a in zip(dict_a[unique_id]['lsoa11_e'], dict_a[unique_id]['start_date']):
                
                # Convert string dates to datetime objects
                if isinstance(date_a, str):
                    date_a = datetime.strptime(date_a, '%Y-%m-%d')
                    
                # Check if address_a is present dict_b
                matching_indices = [
                    idx for idx, addr in enumerate(dict_b[unique_id]['lsoa11cd_e']) if addr == address_a
                ]
                for idx_b in matching_indices:
                    # Get corresponding date from dict_b for matching address
                    date_b = dict_b[unique_id]['start_date'][idx_b]
                    
                    # Convert string dates to datetime objects
                    if isinstance(date_b, str):
                        date_b = datetime.strptime(date_b, '%Y-%m-%d')
                    
                    # calcuate Date differences (days)
                    date_diff = (date_b - date_a).days
                    
                    # update min_date_diff (if multiple matches, and calculate using closest date), mark match to true
                    if min_date_diff is None or date_diff < min_date_diff:
                        min_date_diff = date_diff
                        best_match_row = {
                            'llc_0002_stud_id': unique_id,
                            'lsoa_cohort': address_a,
                            'match': True,
                            'min_date_diff': min_date_diff 
                        }
                if best_match_row:
                    data.append(best_match_row)
                    
    # convert list of dict to dataframe
    df = pd.DataFrame(data)

    return df

In [ ]:
# compare no matching address

def compare_no_matching_addresses(dict_a, dict_b):
    rows = []
    
    shared_ids = set(dict_a.keys()) & set(dict_b.keys())
    for unique_id in shared_ids:
        addresses_a = set(dict_a[unique_id]['lsoa11_e'])
        addresses_b = set(dict_b[unique_id]['lsoa11cd_e'])
        
        if addresses_a.isdisjoint(addresses_b):
            rows.append({
                'llc_0002_stud_id': unique_id,
                'match': False
            })
    df = pd.DataFrame(rows)
    
    return df

In [ ]:
# run the functions
df = compare_dicts_to_df(dict1,dict2)
df2 = compare_no_matching_addresses(dict1, dict2)

In [ ]:
df.describe()


In [ ]:
# inspect df
# notice, current comparing all matching addresses and date difference.
# for each ID, keep the smallest min_date_diff

df['lsoa_cohort'].describe()



In [ ]:
df['min_date_diff_abs'] = df['min_date_diff'].abs()
df = df.loc[df.groupby('llc_0002_stud_id')['min_date_diff_abs'].idxmin()]

In [ ]:
# inspect df

df
df.to_csv("cross_sectional_min_diff.csv")

In [ ]:
df['min_date_diff'].describe()

In [ ]:
# random check
df[df['llc_0002_stud_id'] == "195286939860110"]


In [ ]:
df['lsoa_cohort'].describe()

In [ ]:
# inspect df2 - geopermission, linked data, used NHSD service, but did not have a matching encrypted LSOA
df2

In [ ]:
df2.to_csv("no_cohort_nhsd_match.csv")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Plot: variation from zero - raincloud plot using only seaborn should be fine.

p25 = np.percentile(df['min_date_diff'], 25)
p50 = np.percentile(df['min_date_diff'], 50)
p75 = np.percentile(df['min_date_diff'], 75)

plt.figure(figsize = (10,6))

sns.violinplot(x = 'min_date_diff', data=df, inner = None, color = "White", bw= 0.3, zorder = 2)

sns.boxplot(x ='min_date_diff', data=df, whis = (5,95), width = 0.2, color = 'lightblue', zorder = 3)

y_jittered = np.random.uniform(0.5, 0.05, size = len(df))
plt.scatter(df['min_date_diff'], y_jittered, alpha = 0.02, color = 'grey', edgecolor = "blue", zorder = 1)

plt.annotate('-2689 (P25)', xy = (p25, 0), xytext =(p25, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-743 (P50)', xy = (p50, 0), xytext =(p50, -0.15), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-243 (P75)', xy = (p75, 0), xytext =(p75, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.title('Any Match: Min Date Difference')
plt.xlabel('Date Difference (days)')
plt.grid(True)
plt.yticks([])

plt.show()

In [ ]:
# Plot: variation from zero - raincloud plot using only seaborn should be fine.

df['min_date_diff_year'] = df['min_date_diff']/365.25

p25 = np.percentile(df['min_date_diff_year'], 25)
p50 = np.percentile(df['min_date_diff_year'], 50)
p75 = np.percentile(df['min_date_diff_year'], 75)

plt.figure(figsize = (10,6))

sns.violinplot(x = 'min_date_diff_year', data=df, inner = None, color = "White", bw= 0.3, zorder = 2)

sns.boxplot(x ='min_date_diff_year', data=df, whis = (5,95), width = 0.2, color = 'lightblue', zorder = 3)

y_jittered = np.random.uniform(0.5, 0.05, size = len(df))
plt.scatter(df['min_date_diff_year'], y_jittered, alpha = 0.02, color = 'grey', edgecolor = "blue", zorder = 1)

plt.annotate('-7.4 (P25)', xy = (p25, 0), xytext =(p25, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-2.0 (P50)', xy = (p50, 0), xytext =(p50, -0.15), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-0.7 (P75)', xy = (p75, 0), xytext =(p75, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.title('Any Match: Min Start Date Difference')
plt.xlabel('Date Difference (Years)')
plt.grid(True)
plt.yticks([])

plt.show()

In [ ]:
df['min_date_diff_year'].describe()

In [ ]:
df['lsoa_cohort'].describe()

In [ ]:
df

In [ ]:
## different attempt - no need 2 in cohort, need 2 in NHSD.
## Use the assumption of N-1
## If no next N, set at last available date in NHSD data - 2023-04-06

# keep only if have > 1 'lsoa11_e'
# calculate living periods

from datetime import datetime, timedelta

def calculate_living_periods(dict2):
    result = {}
    default_end_date = datetime.strptime('2023-04-06', '%Y-%m-%d')
    for unique_id, info in dict2.items():
        addresses = info['lsoa11_e']
        start_dates = [datetime.strptime(date, '%Y-%m-%d') for date in info['start_date']]

        #store result
        result[unique_id] = {
            'lsoa11_e': [],
            'start_date': [],
            'end_date_method_n1':[],
            'start_date_median':[],
            'end_date_method_med':[]
        }


        for i in range(len(addresses)):
            current_address = addresses[i]
            current_start_date = start_dates[i]

            # method N-1
            if i + 1 < len(addresses):
                next_start_date = start_dates[i + 1]
                end_date_method_n1 = next_start_date - timedelta(days = 1)
            else:
                end_date_method_n1 = default_end_date

            # method median
            previous_start_date_median = None

            if i + 1 < len(addresses): 
                median_timestamp = (current_start_date.timestamp() + next_start_date.timestamp())/2
                end_date_method_med = datetime.fromtimestamp(median_timestamp)

            else:
                end_date_method_med = default_end_date

            if i == 0:
                start_date_median = current_start_date
            else:
                start_date_median = previous_end_date_median + timedelta(days = 1)

            # results
            result[unique_id]['lsoa11_e'].append(current_address)
            result[unique_id]['start_date'].append(current_start_date.strftime('%Y-%m-%d'))
            result[unique_id]['end_date_method_n1'].append(end_date_method_n1.strftime('%Y-%m-%d'))
            result[unique_id]['end_date_method_med'].append(end_date_method_med.strftime('%Y-%m-%d'))
            result[unique_id]['start_date_median'].append(start_date_median.strftime('%Y-%m-%d'))

            previous_end_date_median = end_date_method_med

    return result




In [ ]:
living_periods_cohort = calculate_living_periods(dict1)

In [ ]:
living_periods_cohort

In [ ]:
# ID 110289585445139 has same start_date both address
# change end_date to be the same.
living_periods_cohort['110289585445139']['end_date_method_n1'] = ['2023-04-06','2023-04-06']
living_periods_cohort['110289585445139']['start_date_median'] = ['2016-11-28', '2016-11-28']
living_periods_cohort['110289585445139']['end_date_method_med'] = ['2023-04-06', '2023-04-06']



In [ ]:
len(living_periods_cohort)

In [ ]:
# output as json

with open("living_periods_cohort.json", "w") as outfile:
    json.dump(living_periods_cohort, outfile)
    

In [ ]:
from datetime import datetime, timedelta

def calculate_living_periods(dict2):
    result = {}
    default_end_date = datetime.strptime('2023-04-06', '%Y-%m-%d')
    for unique_id, info in dict2.items():
        if len(info['lsoa11cd_e']) > 1:
            addresses = info['lsoa11cd_e']
            start_dates = [datetime.strptime(date, '%Y-%m-%d') for date in info['start_date']]
            
            #store result
            result[unique_id] = {
                'lsoa11cd_e': [],
                'start_date': [],
                'end_date_method_n1':[],
                'start_date_median':[],
                'end_date_method_med':[]
            }
            
            
            for i in range(len(addresses)):
                current_address = addresses[i]
                current_start_date = start_dates[i]
                
                # method N-1
                if i + 1 < len(addresses):
                    next_start_date = start_dates[i + 1]
                    end_date_method_n1 = next_start_date - timedelta(days = 1)
                else:
                    end_date_method_n1 = default_end_date
                    
                # method median
                previous_start_date_median = None
                
                if i + 1 < len(addresses): 
                    median_timestamp = (current_start_date.timestamp() + next_start_date.timestamp())/2
                    end_date_method_med = datetime.fromtimestamp(median_timestamp)
                                    
                else:
                    end_date_method_med = default_end_date
                    
                if i == 0:
                    start_date_median = current_start_date
                else:
                    start_date_median = previous_end_date_median + timedelta(days = 1)
                    
                # results
                result[unique_id]['lsoa11cd_e'].append(current_address)
                result[unique_id]['start_date'].append(current_start_date.strftime('%Y-%m-%d'))
                result[unique_id]['end_date_method_n1'].append(end_date_method_n1.strftime('%Y-%m-%d'))
                result[unique_id]['end_date_method_med'].append(end_date_method_med.strftime('%Y-%m-%d'))
                result[unique_id]['start_date_median'].append(start_date_median.strftime('%Y-%m-%d'))
                
                previous_end_date_median = end_date_method_med

    return result

In [ ]:
from datetime import datetime, timedelta

def calculate_living_periods(dict2):
    result = {}
    default_end_date = datetime.strptime('2023-04-06', '%Y-%m-%d')
    for unique_id, info in dict2.items():
        addresses = info['lsoa11cd_e']
        start_dates = [datetime.strptime(date, '%Y-%m-%d') for date in info['start_date']]

        #store result
        result[unique_id] = {
            'lsoa11cd_e': [],
            'start_date': [],
            'end_date_method_n1':[],
            'start_date_median':[],
            'end_date_method_med':[]
        }


        for i in range(len(addresses)):
            current_address = addresses[i]
            current_start_date = start_dates[i]

            # method N-1
            if i + 1 < len(addresses):
                next_start_date = start_dates[i + 1]
                end_date_method_n1 = next_start_date - timedelta(days = 1)
            else:
                end_date_method_n1 = default_end_date

            # method median
            previous_start_date_median = None

            if i + 1 < len(addresses): 
                median_timestamp = (current_start_date.timestamp() + next_start_date.timestamp())/2
                end_date_method_med = datetime.fromtimestamp(median_timestamp)

            else:
                end_date_method_med = default_end_date

            if i == 0:
                start_date_median = current_start_date
            else:
                start_date_median = previous_end_date_median + timedelta(days = 1)

            # results
            result[unique_id]['lsoa11cd_e'].append(current_address)
            result[unique_id]['start_date'].append(current_start_date.strftime('%Y-%m-%d'))
            result[unique_id]['end_date_method_n1'].append(end_date_method_n1.strftime('%Y-%m-%d'))
            result[unique_id]['end_date_method_med'].append(end_date_method_med.strftime('%Y-%m-%d'))
            result[unique_id]['start_date_median'].append(start_date_median.strftime('%Y-%m-%d'))

            previous_end_date_median = end_date_method_med

    return result

In [ ]:
living_periods_nhsd = calculate_living_periods(dict2)

In [ ]:
living_periods_nhsd

In [ ]:
with open("living_periods_nhsd.json", "w") as outfile:
    json.dump(living_periods_nhsd, outfile)

In [ ]:
len(living_periods_nhsd)

In [ ]:
# add new method. end_date_random. 

In [ ]:
def random_date(start_date_str, end_date_str, distribution = 'beta', beta_a = 2, beta_b =  5, seed = None):
    """
    Generate random end date for N1 method.
    - Beta distribution
    
    """
    if seed is None:
        seed = 12345
    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = datetime.strptime(end_date_str, '%Y-%m-%d')
    
    delta_days = (end_date - start_date).days
    
    if delta_days <= 0:
        return start_date_str
    
    if distribution == 'uniform':
        fraction = np.random.uniform(0,1)
    elif distribution == 'beta':
        fraction = np.random.beta(beta_a, beta_b)
    else:
        fraction = np.random.uniform(0,1)
        
    random_days = int(fraction * delta_days)
    random_date_obj = start_date + timedelta(days = random_days)
    
    return random_date_obj.strftime('%Y-%m-%d')

In [ ]:
for unique_id, record in living_periods_nhsd.items():
    if (isinstance(record.get('start_date'), list) and isinstance(record.get('end_date_method_n1'), list)):
        
        new_start_random = []
        new_end_random = []
        n_intervals = len(record['start_date'])
        current_start = record['start_date'][0]
        
        for i in range(n_intervals):
            new_start_random.append(current_start)
            provided_end = record['end_date_method_n1'][i]
            if i == n_intervals-1:
                random_end = provided_end
            else:
                random_end = random_date(current_start, provided_end, distribution = 'beta', beta_a = 5, beta_b = 2, seed = 42)
                # print(current_start, provided_end, random_end)
            new_end_random.append(random_end)
            
            if i < n_intervals - 1:
                current_start = (datetime.strptime(random_end, '%Y-%m-%d') + timedelta(days = 1)).strftime('%Y-%m-%d')
        record['start_date_method_n1_random'] = new_start_random
        record['end_date_method_n1_random'] = new_end_random
    else:
        record['start_date_method_n1_random'] = (record['start_date'] if not isinstance(record['start_date'], list)
                                                else record['start_date'][0])
        record['end_date_method_n1_random'] = (record['end_date_method_n1'] if not isinstance(record['end_date_method_n1'], list)
                                                else record['end_date_method_n1'][0])

        
len(living_periods_nhsd)

In [ ]:
with open("living_periods_nhsd.json", "w") as outfile:
    json.dump(living_periods_nhsd, outfile)

In [ ]:
for unique_id, record in living_periods_cohort.items():
    if (isinstance(record.get('start_date'), list) and isinstance(record.get('end_date_method_n1'), list)):
        
        new_start_random = []
        new_end_random = []
        n_intervals = len(record['start_date'])
        current_start = record['start_date'][0]
        
        for i in range(n_intervals):
            new_start_random.append(current_start)
            provided_end = record['end_date_method_n1'][i]
            if i == n_intervals-1:
                random_end = provided_end
            else:
                random_end = random_date(current_start, provided_end, distribution = 'beta', beta_a = 5, beta_b = 2, seed = 42)
                # print(current_start, provided_end, random_end)
            new_end_random.append(random_end)
            
            if i < n_intervals - 1:
                current_start = (datetime.strptime(random_end, '%Y-%m-%d') + timedelta(days = 1)).strftime('%Y-%m-%d')
        record['start_date_method_n1_random'] = new_start_random
        record['end_date_method_n1_random'] = new_end_random
    else:
        record['start_date_method_n1_random'] = (record['start_date'] if not isinstance(record['start_date'], list)
                                                else record['start_date'][0])
        record['end_date_method_n1_random'] = (record['end_date_method_n1'] if not isinstance(record['end_date_method_n1'], list)
                                                else record['end_date_method_n1'][0])

        
len(living_periods_cohort)

In [ ]:
with open("living_periods_cohort.json", "w") as outfile:
    json.dump(living_periods_cohort, outfile)

In [ ]:
from datetime import datetime
from collections import defaultdict

def calculate_overlap_both(dict_a, dict_b):
    overlap_results = defaultdict(lambda: {'overlap_days_n1': 0, 'total_period_n1': 0,
                                          'overlap_days_med':0, 'total_period_med':0,
                                          'overlap_days_random':0, 'total_period_random':0})
    
    for unique_id, info_a in dict_a.items():
        if unique_id in dict_b:
            info_b = dict_b[unique_id]
            # a = n-1, b = median
            address_map_a = {address: (start_a, end_a, start_b, end_b, start_c, end_c) for address, start_a, end_a, start_b, end_b, start_c, end_c in zip(info_a['lsoa11_e'], info_a['start_date'], info_a['end_date_method_n1'], info_a['start_date_median'], info_a['end_date_method_med'], info_a['start_date_method_n1_random'], info_a['end_date_method_n1_random'])} 
            address_map_b = {address: (start_a, end_a, start_b, end_b, start_c, end_c) for address, start_a, end_a, start_b, end_b, start_c, end_c in zip(info_b['lsoa11cd_e'], info_b['start_date'], info_b['end_date_method_n1'], info_b['start_date_median'], info_b['end_date_method_med'], info_b['start_date_method_n1_random'], info_b['end_date_method_n1_random'])} 

            for address, (start_a1, end_a1, start_a2, end_a2, start_a3, end_a3) in address_map_a.items():
                if address in address_map_b:
                    # get dates for both methods in both dictionaries
                    start_b1, end_b1, start_b2, end_b2, start_b3, end_b3 = address_map_b[address]
                    
                    #parse dates if strings
                    start_a1, end_a1 = datetime.strptime(start_a1, '%Y-%m-%d'), datetime.strptime(end_a1, '%Y-%m-%d')
                    start_a2, end_a2 = datetime.strptime(start_a2, '%Y-%m-%d'), datetime.strptime(end_a2, '%Y-%m-%d')
                    start_a3, end_a3 = datetime.strptime(start_a3, '%Y-%m-%d'), datetime.strptime(end_a3, '%Y-%m-%d')
                    start_b1, end_b1 = datetime.strptime(start_b1, '%Y-%m-%d'), datetime.strptime(end_b1, '%Y-%m-%d')
                    start_b2, end_b2 = datetime.strptime(start_b2, '%Y-%m-%d'), datetime.strptime(end_b2, '%Y-%m-%d')
                    start_b3, end_b3 = datetime.strptime(start_b3, '%Y-%m-%d'), datetime.strptime(end_b3, '%Y-%m-%d')                    
                    # calculate overlap for method a (n-1)
                    total_days_a = (end_a1 - start_a1).days
                    overlap_start_a = max(start_a1, start_b1)
                    overlap_end_a = min(end_a1, end_b1)
                    overlap_length_a = max((overlap_end_a - overlap_start_a).days, 0)
                    
                    """
                    # previous attempt
                    overlap_start_a = max(start_a1, start_b1)
                    overlap_end_a = min(end_a1, end_b1)
                    overlap_length_a = max((overlap_end_a - overlap_start_a).days, 0)
                    # calculate combined time span
                    combined_start_a = min(start_a1, start_b1)
                    combined_end_a = max(end_a1, end_b1)
                    combined_span_a = (combined_end_a - combined_start_a).days
                    
                    overlap_prop_a = overlap_length_a / combined_span_a if combined_span_a > 0 else 0"""
                    
                    # Calculate overlap for Method b (median)
                    total_days_b = (end_a2 - start_a2).days
                    overlap_start_b = max(start_a2, start_b2)
                    overlap_end_b = min(end_a2, end_b2)
                    overlap_length_b = max((overlap_end_b - overlap_start_b).days, 0)
                    
                    # Calculate overlap for method c (Random)
                    total_days_c = (end_a3 - start_a3).days
                    overlap_start_c = max(start_a3, start_b3)
                    overlap_end_c = min(end_a3, end_b3)
                    overlap_length_c = max((overlap_end_c - overlap_start_c).days, 0)
                    
                    # sum overlap legnth for same address
                    overlap_results[(unique_id, address)]['overlap_days_n1'] += overlap_length_a
                    overlap_results[(unique_id, address)]['total_period_n1'] += total_days_a
                    overlap_results[(unique_id, address)]['overlap_days_med'] += overlap_length_b
                    overlap_results[(unique_id, address)]['total_period_med'] += total_days_b
                    overlap_results[(unique_id, address)]['overlap_days_random'] += overlap_length_c
                    overlap_results[(unique_id, address)]['total_period_random'] += total_days_c
                    
    # result
    final_results = []
    for (unique_id, address), lengths in overlap_results.items():
        overlap_prop_a = lengths['overlap_days_n1'] / lengths['total_period_n1'] if lengths['total_period_n1'] > 0 else 0
        overlap_prop_b = lengths['overlap_days_med'] / lengths['total_period_med'] if lengths['total_period_med'] > 0 else 0
        overlap_prop_c = lengths['overlap_days_random'] / lengths['total_period_random'] if lengths['total_period_random'] > 0 else 0
        
        final_results.append({
            'llc_0002_stud_id': unique_id,
            'lsoa11_e': address,
            'total_days_n1':lengths['total_period_n1'],
            'total_days_med':lengths['total_period_med'],
            'total_days_random': lengths['total_period_random'],
            'overlap_prop_n_1': overlap_prop_a,
            'overlap_days_n_1': lengths['overlap_days_n1'],
            'overlap_prop_median': overlap_prop_b,
            'overlap_days_median': lengths['overlap_days_med'],
            'overlap_prop_random': overlap_prop_c,
            'overlap_days_random': lengths['overlap_days_random']
        })
            
    return final_results


In [ ]:
final_results = calculate_overlap_both(living_periods_cohort, living_periods_nhsd)

In [ ]:
overlap_results = pd.DataFrame(final_results)

In [ ]:
overlap_results.describe()

In [ ]:
overlap_results[overlap_results['overlap_prop_n_1']< 1] 

In [ ]:
len(overlap_results['lsoa11_e'].unique())

In [ ]:
# aggregated findings
df_total = overlap_results.groupby(['llc_0002_stud_id'], as_index = False)[['total_days_n1','total_days_med','total_days_random','overlap_days_n_1','overlap_days_median','overlap_days_random']].sum()


In [ ]:
#df_total[df_total['llc_0002_stud_id'] == '199962502428727']

In [ ]:
df_total['overlap_prop_n_1'] = df_total['overlap_days_n_1']/ df_total['total_days_n1']
df_total['overlap_prop_median'] = df_total['overlap_days_median']/ df_total['total_days_med']
df_total['overlap_prop_random'] = df_total['overlap_days_random']/ df_total['total_days_random']                                                                          

In [ ]:
df_total.describe()

In [ ]:
df_total.to_csv("longitudinal_proportion_comparison_norestrict.csv")